# Chapter 2: Running a Forward Pass

> Source PDF: `02_Running_a_Forward_Pass.pdf`  
> Instructor: Maham Faisal Khan, Senior Data Scientist

## Learning Objectives
By the end of this notebook, you will be able to:
- Describe forward and backward passes.
- Build forward passes for binary classification, multiclass classification, and regression.
- Explain why loss functions are needed.
- Use one-hot encoding and cross-entropy loss.
- Compute gradients with backpropagation.
- Update model parameters manually and with an optimizer.
- Write a compact PyTorch training loop.


In [ ]:
# Core imports used throughout this notebook
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader


## 2.1 What Is a Forward Pass?

A **forward pass** moves input data through the network to produce a prediction.

| Task | Typical Final Output |
|---|---|
| Binary classification | One probability between 0 and 1 |
| Multiclass classification | A probability distribution that sums to 1 |
| Regression | Continuous numerical values |


## 2.2 Is There Also a Backward Pass?

Yes. The backward pass, or **backpropagation**, computes gradients used to update weights and biases during training.

A typical training loop repeats: forward pass, compare outputs to ground truth, backpropagate gradients, and update parameters.


## 2.3 Binary Classification: Forward Pass

For binary classification, the model returns one probability per sample. A sigmoid layer constrains each output to `[0, 1]`.


In [ ]:
# Create input data of shape 5 x 6
input_data = torch.tensor([
    [-0.4421,  1.5207,  2.0607, -0.3647,  0.4691,  0.0946],
    [-0.9155, -0.0475, -1.3645,  0.6336, -1.9520, -0.3398],
    [ 0.7406,  1.6763, -0.8511,  0.2432,  0.1123, -0.0633],
    [-1.6630, -0.0718, -0.1285,  0.5396, -0.0288, -0.8622],
    [-0.7413,  1.7920, -0.0883, -0.6685,  0.4745, -0.4245]
])

# Create binary classification model
model = nn.Sequential(
    nn.Linear(6, 4),  # First linear layer
    nn.Linear(4, 1),  # Second linear layer
    nn.Sigmoid()      # Sigmoid activation function
)

# Pass input data through model
output = model(input_data)
print(output)

predicted_classes = (output > 0.5).int()
print("Predicted classes:", predicted_classes.squeeze().tolist())


The output contains five probabilities, one for each row of data. With a threshold of `0.5`, values above `0.5` are labeled class `1`.


## 2.4 Multiclass Classification: Forward Pass

For multiclass classification, the final layer produces one score per class. Softmax turns class scores into probabilities.


In [ ]:
# Specify model has three classes
n_classes = 3

# Create multiclass classification model
model = nn.Sequential(
    nn.Linear(6, 4),          # First linear layer
    nn.Linear(4, n_classes),  # Second linear layer
    nn.Softmax(dim=-1)        # Softmax activation
)

# Pass input data through model
output = model(input_data)
print("Output shape:", output.shape)
print(output)
print("Row sums:", output.sum(dim=-1))
print("Predicted labels:", output.argmax(dim=-1))


| Output Property | Meaning |
|---|---|
| Shape `5 x 3` | Five samples and three class probabilities per sample. |
| Each row sums to one | The row is a probability distribution. |
| Highest probability | Used as the predicted class label. |


## 2.5 Regression: Forward Pass

For regression, the network predicts continuous numerical values. Unlike classification, we usually do **not** apply sigmoid or softmax as the final activation.


In [ ]:
# Create regression model
model = nn.Sequential(
    nn.Linear(6, 4),  # First linear layer
    nn.Linear(4, 1)   # Second linear layer
)

# Pass input data through model
output = model(input_data)
print(output)


## 2.6 Why Do We Need a Loss Function?

A loss function gives feedback during training. It takes model prediction $\hat{y}$ and ground truth $y$, then returns a single float measuring prediction error.

| Prediction Quality | Loss |
|---|---|
| Correct prediction | Low loss |
| Wrong prediction | High loss |


## 2.7 One-Hot Encoding Concepts

One-hot encoding converts a class label into a vector with one `1` and the rest `0`s.

| Class Label | One-Hot Vector for 3 Classes |
|---|---|
| `0` | `[1, 0, 0]` |
| `1` | `[0, 1, 0]` |
| `2` | `[0, 0, 1]` |


In [ ]:
import torch.nn.functional as F

print(F.one_hot(torch.tensor(0), num_classes=3))
print(F.one_hot(torch.tensor(1), num_classes=3))
print(F.one_hot(torch.tensor(2), num_classes=3))

one_hot_numpy = np.array([1, 0, 0])
print("NumPy one-hot example:", one_hot_numpy)


## 2.8 Cross-Entropy Loss in PyTorch

Cross-entropy is commonly used for classification. The PDF example compares model scores against a one-hot target.

> ⚠️ In many PyTorch workflows, `nn.CrossEntropyLoss` is used with raw logits and integer class labels.


In [ ]:
from torch.nn import CrossEntropyLoss

scores = torch.tensor([[-0.1211, 0.1059]])
one_hot_target = torch.tensor([[1, 0]])

criterion = CrossEntropyLoss()
loss = criterion(scores.double(), one_hot_target.double())
print(loss)

# Common PyTorch style: integer target class
class_target = torch.tensor([0])
loss_with_class_target = criterion(scores, class_target)
print("Loss with integer class target:", loss_with_class_target)


### Bringing It All Together

| Input | Meaning |
|---|---|
| `scores` | Model predictions before the final softmax. |
| `one_hot_target` or class index | Ground truth label representation. |
| `loss` | A single float that training tries to minimize. |


## 2.9 Using Derivatives to Update Model Parameters

Gradients tell us the local direction and magnitude of change. Backpropagation calculates gradients from the output layer backward through earlier layers.


In [ ]:
# Create the model and run a forward pass
model = nn.Sequential(
    nn.Linear(16, 8),
    nn.Linear(8, 4),
    nn.Linear(4, 2)
)

sample = torch.randn(1, 16)
target = torch.tensor([0])
prediction = model(sample)

# Calculate the loss and compute the gradients
criterion = CrossEntropyLoss()
loss = criterion(prediction, target)
loss.backward()

# Access each layer's gradients
for layer_index in [0, 1, 2]:
    print(f"Layer {layer_index} weight grad shape:", model[layer_index].weight.grad.shape)
    print(f"Layer {layer_index} bias grad shape:", model[layer_index].bias.grad.shape)


## 2.10 Updating Model Parameters

The conceptual update rule is:

$$\text{new parameter} = \text{old parameter} - \text{learning rate} \times \text{gradient}$$


In [ ]:
# Learning rate is typically small
lr = 0.001

# Conceptual manual update for the first layer
with torch.no_grad():
    weight = model[0].weight
    weight_grad = model[0].weight.grad
    model[0].weight -= lr * weight_grad

    bias = model[0].bias
    bias_grad = model[0].bias.grad
    model[0].bias -= lr * bias_grad

print("Updated first layer parameters manually.")


## 2.11 Gradient Descent and Optimizers

For non-convex functions, neural networks use iterative optimization. The most common beginner optimizer is **stochastic gradient descent (SGD)**.


In [ ]:
import torch.optim as optim

# Create the optimizer
optimizer = optim.SGD(model.parameters(), lr=0.001)

# Optimizer handles updating model parameters after gradients are calculated
optimizer.step()
print("Optimizer step completed")


## 2.12 Writing Our First Training Loop

Training a neural network involves creating a model, choosing a loss function, creating a dataset, defining an optimizer, and running a loop.

The PDF uses a data science salary regression example with `experience_level`, `employment_type`, `remote_ratio`, and `company_size`.


In [ ]:
# Example salary-like regression data
features = np.array([
    [0, 0, 0.5, 1],
    [1, 0, 1.0, 2],
    [2, 0, 0.0, 1],
    [1, 0, 1.0, 0],
    [2, 0, 1.0, 1],
], dtype=np.float32)

target = np.array([[0.036], [0.133], [0.234], [0.076], [0.170]], dtype=np.float32)

# Create the dataset and the dataloader
dataset = TensorDataset(torch.tensor(features).float(), torch.tensor(target).float())
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

# Create the model
model = nn.Sequential(
    nn.Linear(4, 2),
    nn.Linear(2, 1)
)

# Create the loss and optimizer
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.001)


In [ ]:
# Mean squared error loss from first principles
def mean_squared_loss(prediction, target):
    return np.mean((prediction - target) ** 2)

prediction_np = np.array([[0.10], [0.12]])
target_np = np.array([[0.00], [0.20]])
print("Manual MSE:", mean_squared_loss(prediction_np, target_np))

# In PyTorch
prediction = torch.tensor(prediction_np).float()
target_tensor = torch.tensor(target_np).float()
criterion = nn.MSELoss()
loss = criterion(prediction, target_tensor)
print("PyTorch MSE:", loss)


In [ ]:
# The training loop
num_epochs = 10

for epoch in range(num_epochs):
    epoch_loss = 0.0
    for data in dataloader:
        # Set the gradients to zero
        optimizer.zero_grad()

        # Get feature and target from the data loader
        feature, target_batch = data

        # Run a forward pass
        pred = model(feature)

        # Compute loss and gradients
        loss = criterion(pred, target_batch)
        loss.backward()

        # Update the parameters
        optimizer.step()
        epoch_loss += loss.item()

    if epoch in {0, num_epochs - 1}:
        print(f"Epoch {epoch + 1}: mean loss = {epoch_loss / len(dataloader):.6f}")


## Chapter Summary

| Concept | Key Point |
|---|---|
| Forward pass | Computes predictions from inputs. |
| Backward pass | Computes gradients for parameter updates. |
| Binary classification | Uses sigmoid for one probability per sample. |
| Multiclass classification | Uses softmax for one probability distribution per sample. |
| Regression | Predicts continuous values, often with MSE loss. |
| Optimizer | Applies gradient-based parameter updates. |

✅ **PDF coverage:** forward pass, backward pass, binary/multiclass/regression outputs, loss functions, one-hot encoding, cross-entropy, derivatives, backpropagation, parameter updates, SGD, MSE, salary dataset concept, and the training loop.
